# 03 — API → index matrix, under three identities

**What this produces.** For every API in `doc-api-sql-matrix-latest.md`, every SQL
statement it issues, and whether that statement uses an index — driven as
**admin / authorized / unauthorized**, each from a restarted Polaris, then
EXPLAINed with *that case's own parameters* in both index states.

Presence, not performance. Every EXPLAIN here is plan-only: `EXPLAIN ANALYZE` on
an INSERT/UPDATE/DELETE really performs it, so plain EXPLAIN is the only safe way
to ask the question of a write — and it leaves no clock in the output for anyone
to misquote.

Plan: `PLAN-api-index-matrix.md`. Commands: `RUNBOOK-api-index-matrix.md`.

---

### How to run this

**Not `Restart & Run All` on the first pass.** The phases are gated by the flags
in cell 2 and several are slow or destructive. Turn on one phase, run it, read
what it printed, then move to the next. Every phase is separately re-runnable.

The order is load-bearing in two places:

* **admin drives LAST.** Only the admin case mutates — the other two are refused
  before touching anything — so running it last leaves the fixture pristine for
  the cases that must see it unchanged.
* **Polaris restarts before each drive.** A cold `InMemoryEntityCache` yields the
  maximal statement set and is what makes the three cases comparable to each
  other rather than to whichever ran first.

In [4]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import json
import os
import pathlib
import subprocess
import sys
import time
from datetime import datetime

sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from polaris_test_utils import *  # noqa: F403 — init_env, root_token, PG_* globals

init_env("local")                 # default env; never "dev"/"prod" here
require_not_prod("03: seeds ~30k entities, restarts Polaris, toggles an index")

# Second guard, independent of require_not_prod: this notebook is destructive.
assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — 03 is local-only."
)

import api_report as rep
import api_surface as surf
import query_profile as qp

HERE = pathlib.Path.cwd()
RUNS = HERE / "runs"
REPORTS = HERE / "reports"
RUNS.mkdir(exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"Polaris  : {POLARIS_URL}  realm={REALM}")
print(f"Postgres : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"stamp    : {STAMP}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
Polaris  : http://192.168.139.2:8181  realm=POLARIS
Postgres : 192.168.139.2:5432/polaris
stamp    : 20260831-144559


## Cell 2 — the switches

Everything is off. Turn on one phase, run it, read the output, turn it off again.

`POLARIS_USER_SECRET` is the shared secret you set by hand in
`principal_authentication_data` — the seeder discards the credentials
`create_principal` returns, so it cannot be read back from the ledger.

In [18]:
# --- cell 2: switches -------------------------------------------------------
RUN_DUMP        = True   # phase 0 — pg_dump AND prove it restores
RUN_SEED        = False   # phase 1 — ~1000 principals. SLOW (tens of minutes).
RUN_ZEROGRANT   = False   # phase 1e — create the zero-grant principal
RUN_SETUP       = False   # phase 3 — build the probe fixture (once)
RUN_DRIVE       = None    # phase 4 — "unauthorized" | "authorized" | "admin"
RUN_EXPLAIN     = None    # phase 5 — "present" | "absent"
RUN_TOGGLE      = None    # phase 5 — "drop" | "create"  (index, between halves)
RUN_TEARDOWN    = False   # phase 6

# Cluster coordinates. Defaults match reset_realm.sh / capture.sh.
NS      = os.environ.get("POLARIS_K8S_NAMESPACE", "datahub-hynix")
DEPLOY  = os.environ.get("POLARIS_K8S_DEPLOYMENT", "benchmarks-polaris")
PGPOD   = os.environ.get("PGPOD", "benchmarks-postgresql-postgresql-ha-postgresql-1")
SCHEMA  = os.environ.get("PG_SCHEMA", "polaris_schema")

# The zero-grant principal. Filled in by cell 6; paste them back here if you
# re-open the notebook, since its secret is NOT the shared one.
ZG_PRINCIPAL = "zerograve_principal"
ZG_ROLE      = "zerograve_principal_role"
ZG_CLIENT_ID = os.environ.get("ZG_CLIENT_ID", "")
ZG_SECRET    = os.environ.get("ZG_SECRET", "")

assert PGPOD.endswith("-1"), (
    f"{PGPOD} — the PRIMARY is -1 here, not -0. Earlier manual resets hit a "
    "read-only replica and appeared to succeed."
)
print("switches read. All phases default to OFF.")

switches read. All phases default to OFF.


## Cell 3 — helpers

`sh()` streams output and **raises on a non-zero exit** rather than returning it
quietly: a phase that half-failed and printed nothing is how a bad fixture
reaches the measurement.

`restart_polaris()` carries two recorded scars. `kubectl rollout restart` only
patches the template, so `rollout status` immediately after can report the
*previous* rollout complete; and readiness means **8181 answers a token**, not
8182 `/q/health/ready` — the port the drive actually uses.

In [7]:
# --- cell 3: helpers --------------------------------------------------------
def sh(cmd, check=True, quiet=False):
    """Run a shell command, stream it, raise on failure."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet:
        if p.stdout:
            print(p.stdout.rstrip())
        if p.stderr:
            print(p.stderr.rstrip(), file=sys.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p


def psql(sql, db=None, quiet=False):
    """One SQL statement through psql -tAc, on the primary.

    THE SQL IS FLATTENED TO ONE LINE FIRST, and that is not cosmetic. `json.dumps`
    escapes a newline as the two characters \\ and n; the shell passes them
    through literally, and psql reads a leading `\\n` as a META-COMMAND, failing
    with `invalid command \\nSELECT`. So any multi-line query silently became a
    psql syntax error rather than a result. Whitespace is nothing to SQL, so
    collapsing it costs nothing and removes the whole class.

    `shlex.quote` rather than `json.dumps`: the values here contain single quotes
    (`LIKE 'zerograve%'`) and json-quoting them is not shell-quoting them.
    """
    import shlex

    db = db or PG_DB
    one_line = " ".join(sql.split())
    return sh(
        f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
        f"psql -U {PG_USER} -d {db} -tAc {shlex.quote(one_line)}",
        quiet=quiet,
    ).stdout.strip()


def polaris_token_ok():
    p = sh(
        f"curl -sf -o /dev/null -X POST {POLARIS_URL}/api/catalog/v1/oauth/tokens "
        f"-d 'grant_type=client_credentials&client_id={ROOT_CLIENT}"
        f"&client_secret={ROOT_SECRET}&scope=PRINCIPAL_ROLE:ALL'",
        check=False, quiet=True,
    )
    return p.returncode == 0


def restart_polaris(timeout=300):
    """Restart Polaris and wait until it really serves — cold entity cache."""
    before = sh(
        f"kubectl get deploy/{DEPLOY} -n {NS} -o jsonpath='{{.status.observedGeneration}}'",
        quiet=True,
    ).stdout.strip()
    sh(f"kubectl rollout restart deploy/{DEPLOY} -n {NS}")
    # `rollout restart` only PATCHES the template. Waiting on rollout status
    # immediately after can report the PREVIOUS rollout complete, so wait for
    # the new generation to be observed first.
    for _ in range(60):
        now = sh(
            f"kubectl get deploy/{DEPLOY} -n {NS} "
            "-o jsonpath='{.status.observedGeneration}'", quiet=True,
        ).stdout.strip()
        if now != before:
            break
        time.sleep(2)
    sh(f"kubectl rollout status deploy/{DEPLOY} -n {NS} --timeout={timeout}s")
    # Ready pods are not a serving Polaris. The gate is a TOKEN on 8181 — the
    # port the drive uses — not 8182 /q/health/ready.
    for i in range(timeout // 3):
        if polaris_token_ok():
            print(f"polaris serving tokens after {i * 3}s")
            return True
        time.sleep(3)
    raise RuntimeError("Polaris never answered a token on 8181")


def volume():
    tables = ["entities", "grant_records", "policy_mapping_record",
              "principal_authentication_data"]
    return {t: int(psql(f"SELECT count(*) FROM {SCHEMA}.{t}", quiet=True))
            for t in tables}


def index_present():
    got = psql(
        "SELECT count(*) FROM pg_indexes WHERE indexname="
        "'idx_grant_records_grantee'", quiet=True,
    )
    return got == "1"


print("helpers defined")

helpers defined


## Cell 4 — Phase 0: dump, and prove the dump restores

A dump nobody has restored is a belief, not a backup. This restores it into a
throwaway database and counts a table before dropping it again.

Plain EXPLAIN does not execute, so the write sweep should not mutate anything —
but this is the first time this audit points EXPLAIN at INSERT/UPDATE/DELETE at
all, and a restore point costs minutes.

In [19]:
# --- cell 4: phase 0 --------------------------------------------------------
if RUN_DUMP:
    dump = f"/tmp/polaris-preexplain-{STAMP}.dump"
    sh(f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
       f"pg_dump -U {PG_USER} -Fc -d {PG_DB} -f {dump}")
    local = pathlib.Path.home() / f"polaris-preexplain-{STAMP}.dump"
    sh(f"kubectl cp {NS}/{PGPOD}:{dump} {local}")
    print(f"\ndump copied to {local}  ({local.stat().st_size / 1e6:.1f} MB)")

    print("\n--- proving it restores ---")
    sh(f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
       f"dropdb -U {PG_USER} --if-exists restoretest", check=False)
    sh(f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
       f"createdb -U {PG_USER} restoretest")
    sh(f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
       f"pg_restore -U {PG_USER} -d restoretest {dump}", check=False)
    n = psql(f"SELECT count(*) FROM {SCHEMA}.grant_records", db="restoretest")
    print(f"restored grant_records: {n}")
    assert int(n) > 0, "the restore produced an empty table — the dump is not a backup"
    sh(f"kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD={PG_PASSWORD} "
       f"dropdb -U {PG_USER} restoretest")

    print(f"\nRESTORE COMMAND, should you need it:\n"
          f"  kubectl cp {local} {NS}/{PGPOD}:{dump}\n"
          f"  kubectl exec -n {NS} {PGPOD} -- env PGPASSWORD=... \\\n"
          f"      pg_restore -U {PG_USER} -d {PG_DB} --clean --if-exists {dump}")
else:
    print("RUN_DUMP is off")

tar: Removing leading `/' from member names

dump copied to /Users/kade/polaris-preexplain-20260831-144559.dump  (0.8 MB)

--- proving it restores ---
60788
restored grant_records: 60788

RESTORE COMMAND, should you need it:
  kubectl cp /Users/kade/polaris-preexplain-20260831-144559.dump datahub-hynix/benchmarks-postgresql-postgresql-ha-postgresql-1:/tmp/polaris-preexplain-20260831-144559.dump
  kubectl exec -n datahub-hynix benchmarks-postgresql-postgresql-ha-postgresql-1 -- env PGPASSWORD=... \
      pg_restore -U polaris -d polaris --clean --if-exists /tmp/polaris-preexplain-20260831-144559.dump


## Cell 5 — Phase 1: seed

**Slow — tens of minutes.** Sized so each table's statements are readable rather
than trivial: without the views and generic tables `entities` lands near 6,000,
inside the band where `entities_pkey` / `idx_entities` can cover a query by
accident of width. That is the 02c failure — at 3,064 `grant_records` the primary
key covered the grantee query and the Seq Scan the audit existed for never
appeared.

`--probe-policy` runs first because policies are feature-flagged in 1.3 and have
seeded as **zero** here before, while `GET /policies` answered 200.

In [9]:
# --- cell 5: phase 1 --------------------------------------------------------
if RUN_SEED:
    print("=== which policy type does this build accept? ===")
    sh("uv run python seed_polaris.py --probe-policy", check=False)

    print("\n=== main fixture: 1000 principals ===")
    sh("uv run python seed_polaris.py --users 1000 --views-per-namespace 5 "
       "--generic-tables-per-namespace 5 --policies-per-namespace 2")

    print("\n=== admin tier: service_admin ===")
    sh("uv run python seed_polaris.py --users 5 --prefix admin --service-admin")

    print("\n=== authorized tier ===")
    sh("uv run python seed_polaris.py --users 100 --prefix authz "
       "--views-per-namespace 5 --generic-tables-per-namespace 5")
    print("\nmeasured volume:", volume())
else:
    print("RUN_SEED is off")

RUN_SEED is off


## Cell 6 — Phase 1e: the zero-grant principal

`seed_polaris` always grants, so this one is built by hand. Its own
`clientSecret` is used directly, so no `principal_authentication_data` edit is
needed.

**It is verified to hold zero.** A principal that accidentally holds grants would
make the unauthorized case a second copy of the authorized one — and the two
would look perfectly consistent.

In [10]:
# --- cell 6: phase 1e -------------------------------------------------------
if RUN_ZEROGRANT:
    from polaris_rest import PolarisREST

    _pc = PolarisREST(POLARIS_URL, REALM, token=root_token())
    r = _pc.create_principal(ZG_PRINCIPAL)
    print("create_principal:", r.status_code)
    body = r.json() if r.status_code < 300 else {}
    creds = body.get("credentials", body) or {}
    ZG_CLIENT_ID = creds.get("clientId", "")
    ZG_SECRET = creds.get("clientSecret", "")
    print("  clientId    :", ZG_CLIENT_ID or "(already exists — recreate to get one)")
    print("  clientSecret:", ZG_SECRET or "(already exists — recreate to get one)")

    print("create_principal_role:", _pc.create_principal_role(ZG_ROLE).status_code)
    print("assign:", _pc.assign_principal_role_to_principal(
        ZG_PRINCIPAL, ZG_ROLE).status_code)
    print("\n>>> GRANT NOTHING. <<<")

    held = psql(
        f"SELECT count(*) FROM {SCHEMA}.grant_records g "
        f"JOIN {SCHEMA}.entities e ON e.id = g.grantee_id "
        f"WHERE e.name LIKE 'zerograve%'"
    )
    print(f"\ngrants held by zerograve*: {held}")
    print("  (only its own principal-role chain is acceptable; anything more and")
    print("   the unauthorized case is a second copy of the authorized one)")
else:
    print("RUN_ZEROGRANT is off")

RUN_ZEROGRANT is off


## Cell 7 — Phase 2: archive the current matrix as a **tracked** file

`reports/.gitignore` is `*.md` with `!doc-*-latest.md`, so **only
`doc-api-sql-matrix-latest.md` is tracked**. Its timestamped twin is
byte-identical but ignored, and a regeneration overwrites `-latest` in place.

This cell only *checks* and prints the commands — it never runs `git`.

In [11]:
# --- cell 7: phase 2 --------------------------------------------------------
_latest = REPORTS / "doc-api-sql-matrix-latest.md"
_pin = REPORTS / "doc-api-sql-matrix-20260820-172931.md"
print("latest exists:", _latest.exists(), "| pin exists:", _pin.exists())
if _latest.exists() and _pin.exists():
    import hashlib
    h = lambda p: hashlib.md5(p.read_bytes()).hexdigest()
    print("identical:", h(_latest) == h(_pin))
tracked = sh("git ls-files reports/ | grep matrix || true", quiet=True).stdout.strip()
print("tracked matrix files:\n ", tracked or "(none)")
print("\nIf the pin is not tracked, run this in a terminal BEFORE regenerating:")
print("  cd reports && printf '!doc-api-sql-matrix-20260820-172931.md\\n' >> .gitignore")
print("  git add -f doc-api-sql-matrix-20260820-172931.md .gitignore")
print("  git commit -m 'Pin the 2026-08-20 root-driven matrix before regenerating'")

latest exists: True | pin exists: True
identical: True
tracked matrix files:
  reports/doc-api-sql-matrix-20260820-172931.md
reports/doc-api-sql-matrix-latest.md

If the pin is not tracked, run this in a terminal BEFORE regenerating:
  cd reports && printf '!doc-api-sql-matrix-20260820-172931.md\n' >> .gitignore
  git add -f doc-api-sql-matrix-20260820-172931.md .gitignore
  git commit -m 'Pin the 2026-08-20 root-driven matrix before regenerating'


## Cell 8 — Phase 3: build the probe fixture, once, as admin

All three cases must drive the **same** fixture, or they differ by more than the
identity and the comparison stops meaning anything. The names are written to
`runs/api-surface-fixture.json` so the later cells reuse them.

In [12]:
# --- cell 8: phase 3 --------------------------------------------------------
if RUN_SETUP:
    sh("uv run python drive_api_surface.py --setup")
else:
    _state = RUNS / "api-surface-fixture.json"
    print("RUN_SETUP is off.",
          f"existing fixture: {json.loads(_state.read_text())['cat']}"
          if _state.exists() else "no fixture recorded yet — turn RUN_SETUP on")

RUN_SETUP is off. existing fixture: apiprofile1788153994_cat


## Cell 9 — Phase 4: one drive

Set `RUN_DRIVE` in cell 2 to a case, run this, then set the next.
**Order: `unauthorized` → `authorized` → `admin` last.**

Each run restarts Polaris (cold entity cache), enables PG statement logging,
rotates into a fresh capture directory, drives all 43 operations, then verifies
the capture actually recorded something before trusting the pass.

`capture.sh rotate` alone does **not** enable statement logging — it is
stop-then-start. `pgon` is separate, and that mistake once cost a clean
9,000-request pass.

In [13]:
# --- cell 9: phase 4 --------------------------------------------------------
if RUN_DRIVE:
    assert RUN_DRIVE in ("unauthorized", "authorized", "admin"), RUN_DRIVE
    cap = f"capture-{RUN_DRIVE}-{STAMP}"

    print(f"=== restarting Polaris (cold cache) before '{RUN_DRIVE}' ===")
    restart_polaris()

    print("\n=== statement logging ON, fresh capture dir ===")
    sh("./capture.sh pgon")
    sh(f"./capture.sh rotate {cap}")

    print(f"\n=== driving 43 operations as '{RUN_DRIVE}' ===")
    extra = ""
    if RUN_DRIVE == "unauthorized":
        assert ZG_CLIENT_ID and ZG_SECRET, (
            "the zero-grant principal's clientId/clientSecret are needed — run "
            "cell 6, or paste them into ZG_CLIENT_ID / ZG_SECRET in cell 2"
        )
        extra = f" --client-id {ZG_CLIENT_ID} --principal-role {ZG_ROLE}"
        os.environ["POLARIS_USER_SECRET"] = ZG_SECRET
    else:
        assert os.environ.get("POLARIS_USER_SECRET"), (
            "POLARIS_USER_SECRET is unset — it is the shared secret you set by "
            "hand in principal_authentication_data"
        )
    sh(f"uv run python drive_api_surface.py --drive --case {RUN_DRIVE} "
       f"--capture {cap}{extra}")

    print(f"\n=== is the capture real? ===")
    # A clean 9,000-request pass once produced a 410-byte capture holding one
    # line: "Apache Polaris Server stopped". Check content, not just size.
    for f in ("polaris.log", "pg-0.log"):
        p = pathlib.Path(cap) / f
        print(f"  {f}: {p.stat().st_size:,} bytes" if p.exists() else f"  {f}: MISSING")
else:
    print("RUN_DRIVE is None — set it to a case in cell 2")

RUN_DRIVE is None — set it to a case in cell 2


## Cell 10 — Phase 5: EXPLAIN one case, one index state

`--index-state` is checked against live `pg_indexes` and the run **refuses** on a
mismatch: EXPLAIN replays against the database as it *is*, so without the check
you get one document holding two cluster states.

Run all three cases at the current state, then use cell 11 to toggle, then run
all three again.

In [14]:
# --- cell 10: phase 5 -------------------------------------------------------
if RUN_EXPLAIN:
    assert RUN_EXPLAIN in ("present", "absent"), RUN_EXPLAIN
    live = "present" if index_present() else "absent"
    print(f"live pg_indexes says: {live}   |  you asked for: {RUN_EXPLAIN}")
    assert live == RUN_EXPLAIN, (
        "refusing before the sweep — planning now would staple these plans onto "
        "a drive taken in the other state, and the report would not say so"
    )
    for case in ("unauthorized", "authorized", "admin"):
        hits = sorted(REPORTS.glob(f"doc-api-sql-matrix-{case}-*.md"))
        if not hits:
            print(f"  ! no report for {case} yet — drive it first")
            continue
        latest = hits[-1]
        print(f"\n=== {case}  ({latest.name}) ===")
        sh(f"uv run python explain_api_matrix.py --case {case} "
           f"--index-state {RUN_EXPLAIN} --matrix {latest.relative_to(HERE)}")
else:
    print("RUN_EXPLAIN is None — set it to 'present' or 'absent' in cell 2")

RUN_EXPLAIN is None — set it to 'present' or 'absent' in cell 2


## Cell 11 — toggle the index between the two halves

`ANALYZE` after each toggle is **not optional**: a planner working from stale
`reltuples` chooses a path for a table size that no longer exists.

In [15]:
# --- cell 11: index toggle --------------------------------------------------
if RUN_TOGGLE:
    print("before:", "present" if index_present() else "absent")
    if RUN_TOGGLE == "drop":
        sh("uv run python drop_grantee_index.py")
    elif RUN_TOGGLE == "create":
        psql("CREATE INDEX CONCURRENTLY idx_grant_records_grantee "
             f"ON {SCHEMA}.grant_records (realm_id, grantee_catalog_id, grantee_id)")
    else:
        raise ValueError(RUN_TOGGLE)
    psql(f"ANALYZE {SCHEMA}.grant_records")
    print("after :", "present" if index_present() else "absent", "| ANALYZEd")
else:
    print("RUN_TOGGLE is None. Current index state:",
          "present" if index_present() else "absent")

RUN_TOGGLE is None. Current index state: absent


## Cell 12 — results so far

Reads whatever `runs/apiexplain-*.json` exist and renders the answer: per case
and index state, how many of the 43 APIs sequentially scan something, and which
relations. This is the table the final report is built from.

In [16]:
# --- cell 12: results -------------------------------------------------------
import pandas as pd

rows = []
for f in sorted(RUNS.glob("apiexplain-*.json")):
    d = json.loads(f.read_text())
    per = d.get("per_api", {})
    seq_apis = [a for a, r in per.items() if r.get("seq_scanned")]
    rels = sorted({x for r in per.values() for x in r.get("seq_scanned", [])})
    rows.append({
        "case": d["case"],
        "index": d["index_state"],
        "APIs": len(per),
        "seq-scanning APIs": len(seq_apis),
        "relations seq-scanned": ", ".join(rels) or "—",
        "pairs": d.get("pairs"),
        "grant_records rows": (d.get("volume") or {}).get("grant_records"),
        "run": f.name,
    })

if rows:
    df = pd.DataFrame(rows).sort_values(["case", "index"])
    display(df)
else:
    print("no apiexplain-*.json yet")

no apiexplain-*.json yet


## Cell 13 — Phase 6: teardown

Removes the probe fixture and sweeps any `apiprofile*` catalog an aborted run
left behind. Only run this once all six EXPLAIN passes exist — the reports are
already on disk, but the fixture is what the parameters point at.

In [17]:
# --- cell 13: phase 6 -------------------------------------------------------
if RUN_TEARDOWN:
    sh("uv run python drive_api_surface.py --teardown")
    print("\nremaining index state:", "present" if index_present() else "absent")
    print("If it is 'absent', restore it — cell 11 with RUN_TOGGLE='create'.")
else:
    print("RUN_TEARDOWN is off")

RUN_TEARDOWN is off


## Cell 14 — cleanup

Explicit disposal, per the repo's notebook rules: the local Pgpool-II pooler is
easy to overload with connections a notebook forgot to close.

In [ ]:
# --- cell 14: cleanup -------------------------------------------------------
for _name in ("df",):
    if _name in dir():
        del globals()[_name]
for _v in list(globals().values()):
    for _m in ("close", "dispose"):
        if hasattr(_v, _m) and _v.__class__.__name__ in ("connection", "Engine"):
            try:
                getattr(_v, _m)()
            except Exception:
                pass
gc.collect()
print("cleaned up")

In [22]:
print(psql(f"SELECT name, type_code FROM {SCHEMA}.entities WHERE name LIKE 'polprobe%'"))
sh("curl -s -H \"Authorization: Bearer $(uv run python -c \\\"import sys;sys.path.insert(0,'../../src');from polaris_test_utils import *;init_env('local');print(root_token())\\\")\" "
   f"{POLARIS_URL}/api/catalog/v1/user1_catalog/namespaces/ns1/policies", check=False)

polprobe1|10
polprobe3|10
polprobe4|10
polprobe1|10
polprobe3|10
polprobe4|10
polprobe1|10
polprobe3|10
polprobe4|10
polprobe1|10
polprobe3|10
polprobe4|10
{"error":{"message":"Missing or invalid realm","type":"MissingOrInvalidRealm","code":404}}


/bin/sh: command substitution: line 0: syntax error near unexpected token `0,'../../src''
/bin/sh: command substitution: line 0: `uv run python -c \"import sys;sys.path.insert(0,'../../src');from polaris_test_utils import *;init_env('local');print(root_token())\"'


CompletedProcess(args='curl -s -H "Authorization: Bearer $(uv run python -c \\"import sys;sys.path.insert(0,\'../../src\');from polaris_test_utils import *;init_env(\'local\');print(root_token())\\")" http://192.168.139.2:8181/api/catalog/v1/user1_catalog/namespaces/ns1/policies', returncode=0, stdout='{"error":{"message":"Missing or invalid realm","type":"MissingOrInvalidRealm","code":404}}', stderr='/bin/sh: command substitution: line 0: syntax error near unexpected token `0,\'../../src\'\'\n/bin/sh: command substitution: line 0: `uv run python -c \\"import sys;sys.path.insert(0,\'../../src\');from polaris_test_utils import *;init_env(\'local\');print(root_token())\\"\'\n')